# Bulk RNA-seq Explorer

HTML report의 interactive 버전.

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(yaml); library(readr); library(dplyr); library(tidyr)
  library(tibble); library(ggplot2); library(scales)
})

`%||%` <- function(a, b) if (is.null(a)) b else a

PROJECT_ROOT <- Sys.getenv("PROJECT_ROOT", normalizePath("..", mustWork = TRUE))
CONFIG_PATH  <- file.path(PROJECT_ROOT, "config", "config.yaml")

rp <- function(...) file.path(PROJECT_ROOT, ...)

cfg       <- yaml::read_yaml(CONFIG_PATH)
samples   <- read_tsv(rp(cfg$input$samples_tsv   %||% "config/samples.tsv"),   show_col_types = FALSE)
contrasts <- read_tsv(rp(cfg$input$contrasts_tsv %||% "config/contrasts.tsv"), show_col_types = FALSE)

no_results <- function(msg = "No results available for this section.") {
  message(msg); invisible(NULL)
}

safe_read <- function(path, sep = ",") {
  if (is.null(path) || !file.exists(path)) return(NULL)
  df <- tryCatch(
    if (sep == "\t") read_tsv(path, show_col_types = FALSE)
    else             read_csv(path, show_col_types = FALSE),
    error = function(e) NULL)
  if (is.null(df) || nrow(df) == 0) return(df)
  df
}

contrast_row <- function(cid) {
  out <- contrasts[contrasts$contrast_id == cid, , drop = FALSE]
  if (nrow(out) == 0) return(NULL)
  as.list(out[1, ])
}

# 탐색할 contrast 선택
CONTRAST <- contrasts$contrast_id[1]
cat("Project :", basename(PROJECT_ROOT),
    "\nContrast:", CONTRAST,
    "\nAvailable:", paste(contrasts$contrast_id, collapse = ", "), "\n")

## 1. QC

MultiQC로 집계한 QC 지표 (`results/qc/qc_summary.tsv`). 샘플 품질(분해, 오염, 라이브러리 이상) 평가.

In [ ]:
qc <- safe_read(rp("results", "qc", "qc_summary.tsv"), sep = "\t")
qc

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.5)
if (is.null(qc) || nrow(qc) == 0 || !"assigned_reads" %in% names(qc)) {
  no_results("Per-sample assigned-read totals not available.")
} else {
  df <- qc |>
    dplyr::mutate(sample    = factor(sample, levels = sample),
                  condition = as.character(condition))
  ggplot(df, aes(x = sample, y = assigned_reads, fill = condition)) +
    geom_col() +
    scale_y_continuous(labels = scales::comma) +
    labs(x = NULL, y = "Assigned reads (sum of counts)") +
    theme_minimal(base_size = 11) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)
if (is.null(qc) || nrow(qc) == 0) {
  no_results("QC rate metrics not available.")
} else {
  rate_cols <- intersect(
    c("mapping_rate", "uniquely_mapped_pct", "rrna_pct",
      "duplication", "dupradar_slope", "exonic_pct"),
    names(qc)
  )
  rate_cols <- rate_cols[vapply(rate_cols, function(c) any(!is.na(qc[[c]])), logical(1))]
  if (length(rate_cols) == 0) {
    no_results("All rate metrics are NA.")
  } else {
    df <- qc |>
      dplyr::select(sample, condition, dplyr::all_of(rate_cols)) |>
      tidyr::pivot_longer(dplyr::all_of(rate_cols), names_to = "metric", values_to = "value") |>
      dplyr::mutate(sample = factor(sample, levels = unique(sample)))
    ggplot(df, aes(x = sample, y = value, fill = condition)) +
      geom_col() +
      facet_wrap(~ metric, scales = "free_y") +
      labs(x = NULL, y = NULL) +
      theme_minimal(base_size = 11) +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
  }
}

### 샘플 처리 지표

Cohort 내 한 샘플이 다음 지표에서 outlier로 나타나면 해당 샘플의 처리 단계를 점검:

- `gene_body_5_3_bias` — RNA 무결성 지표, 1.0 부근은 정상, 0.5 미만은 transcript truncation에 의한 3' bias.
- `intronic_pct`, `intergenic_pct` — pre-mRNA 과다 또는 genomic DNA carryover.
- `insert_size_median` — fragmentation 지표.
- `gc_content_pct` — 외래 종 오염 또는 adapter dimer.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)
if (is.null(qc) || nrow(qc) == 0) {
  no_results("Handling-indicator metrics not available.")
} else {
  cols <- intersect(
    c("gene_body_5_3_bias", "intronic_pct", "intergenic_pct",
      "insert_size_median", "gc_content_pct"),
    names(qc)
  )
  cols <- cols[vapply(cols, function(c) any(!is.na(qc[[c]])), logical(1))]
  if (length(cols) == 0) {
    no_results("Handling indicators all NA — confirm MultiQC modules present.")
  } else {
    df <- qc |>
      dplyr::select(sample, condition, dplyr::all_of(cols)) |>
      tidyr::pivot_longer(dplyr::all_of(cols),
                          names_to = "metric", values_to = "value") |>
      dplyr::mutate(sample = factor(sample, levels = unique(sample)))
    print(
      ggplot(df, aes(x = sample, y = value, fill = condition)) +
        geom_col() +
        facet_wrap(~ metric, scales = "free_y") +
        labs(x = NULL, y = NULL) +
        theme_minimal(base_size = 11) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    )
  }
}

if (!is.null(qc) && "strandedness" %in% names(qc) && any(!is.na(qc$strandedness))) {
  calls <- unique(stats::na.omit(qc$strandedness))
  if (length(calls) == 1) {
    cat(sprintf("Strandedness sanity check: all samples called as %s.\n", calls))
  } else {
    tab <- table(qc$strandedness)
    cat("Strandedness mismatch across samples:",
        paste(sprintf("%s (n=%d)", names(tab), as.integer(tab)), collapse = ", "),
        "\n")
  }
}


## 2. Exploratory

VST 상위 500개 유전자 사용. 계산된 prcomp 객체를 `results/exploratory/pca.rds`에서 로드함.

In [ ]:
pca_obj <- readRDS(rp("results", "exploratory", "pca.rds"))
cor_obj <- readRDS(rp("results", "exploratory", "sample_correlation.rds"))
str(pca_obj, max.level = 1); cat("---\n"); str(cor_obj, max.level = 1)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 5.5)
scores <- pca_obj$scores
ve     <- pca_obj$var_explained
pc1_lbl <- sprintf("PC1 (%.1f%%)", 100 * (ve[["PC1"]] %||% NA))
pc2_lbl <- sprintf("PC2 (%.1f%%)", 100 * (ve[["PC2"]] %||% NA))

ggplot(scores, aes(x = PC1, y = PC2, colour = condition, shape = as.factor(batch), label = sample)) +
  geom_point(size = 3) +
  geom_text(aes(label = sample), nudge_y = 0.04, size = 3, show.legend = FALSE) +
  labs(x = pc1_lbl, y = pc2_lbl, colour = "condition", shape = "batch") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 4)
ve <- pca_obj$var_explained
scree <- data.frame(PC = factor(names(ve), levels = names(ve)),
                    pct = as.numeric(ve) * 100)
scree <- head(scree, min(10, nrow(scree)))

ggplot(scree, aes(x = PC, y = pct)) +
  geom_col(fill = "#4c7fb0") +
  labs(x = NULL, y = "% variance explained") +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 6.5)
M   <- cor_obj$spearman
ord <- if (!is.null(cor_obj$hclust)) cor_obj$hclust$order else seq_len(nrow(M))
M   <- M[ord, ord, drop = FALSE]
df  <- as.data.frame(as.table(M))
names(df) <- c("sample_x", "sample_y", "rho")
df$sample_x <- factor(df$sample_x, levels = rownames(M))
df$sample_y <- factor(df$sample_y, levels = rownames(M))

ggplot(df, aes(x = sample_x, y = sample_y, fill = rho)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.3f", rho)), size = 3, colour = "grey20") +
  scale_fill_gradient(low = "#ffffff", high = "#4c7fb0",
                      limits = c(min(df$rho, na.rm = TRUE), 1)) +
  labs(x = NULL, y = NULL, fill = expression(rho)) +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4.5)
hc <- cor_obj$hclust
op <- graphics::par(mar = c(6, 1, 1, 1))
plot(hc, main = "", xlab = "", sub = "", ylab = "", yaxt = "n", hang = -1)
graphics::par(op)

## 3. Differential expression — `CONTRAST`

DESeq2 결과. 다음 셀의 `padj_cutoff` / `lfc_cutoff` 값을 바꿔 DEG cutoff를 조절. LFC는 apeglm 적용, padj는 B-H method 사용하였음.

In [ ]:
padj_cutoff <- cfg$de$primary$padj    %||% 0.05
lfc_cutoff  <- cfg$de$primary$abs_lfc %||% 1

de_tbl  <- safe_read(rp("results", "de", CONTRAST, "deseq2_results.csv"))
deg_sum <- safe_read(rp("results", "de", CONTRAST, "deg_summary.tsv"), sep = "\t")

dds_path <- rp("results", "exploratory", "dds_vst.rds")
vst_mat  <- if (file.exists(dds_path)) {
  obj <- readRDS(dds_path)
  tryCatch(SummarizedExperiment::assay(obj), error = function(e) as.matrix(obj@assays@data[[1]]))
} else NULL

deg_sum

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 6)
v_df <- de_tbl |>
  dplyr::mutate(neglog10p = -log10(pmax(pvalue, 1e-300)),
                sig = !is.na(padj) & padj < padj_cutoff & abs(log2FoldChange) >= lfc_cutoff)

sig_df <- v_df |> dplyr::filter(sig)
ns_df  <- v_df |> dplyr::filter(!sig)
if (nrow(ns_df) > 3000) { set.seed(42); ns_df <- ns_df[sample.int(nrow(ns_df), 3000), ] }

ggplot() +
  geom_point(data = ns_df,  aes(log2FoldChange, neglog10p),
             colour = "grey70", alpha = 0.45, size = 1.1) +
  geom_point(data = sig_df, aes(log2FoldChange, neglog10p),
             colour = "#c0392b", alpha = 0.85, size = 1.6) +
  geom_vline(xintercept = c(-lfc_cutoff, lfc_cutoff), linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = -log10(padj_cutoff),       linetype = "dashed", colour = "grey40") +
  labs(title = CONTRAST, x = "log2FoldChange (apeglm)", y = "-log10(pvalue)",
       subtitle = sprintf("sig: padj<%.2g & |LFC|>=%.2g", padj_cutoff, lfc_cutoff)) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 5.5)
ma_df <- de_tbl |>
  dplyr::mutate(sig = !is.na(padj) & padj < padj_cutoff & abs(log2FoldChange) >= lfc_cutoff)

ggplot(ma_df, aes(log10(pmax(baseMean, 1)), log2FoldChange, colour = sig)) +
  geom_point(alpha = 0.5, size = 1.2) +
  scale_colour_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "grey60")) +
  geom_hline(yintercept = 0, colour = "grey40") +
  labs(x = "log10(baseMean)", y = "log2FoldChange (apeglm)",
       colour = sprintf("padj<%.2g & |LFC|>=%.2g", padj_cutoff, lfc_cutoff)) +
  theme_minimal(base_size = 12)

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 9)
top <- de_tbl |>
  dplyr::filter(!is.na(padj), !is.na(log2FoldChange)) |>
  dplyr::arrange(padj) |>
  head(30)

rows <- intersect(top$gene_id, rownames(vst_mat))
M <- vst_mat[rows, , drop = FALSE]
label_map <- setNames(top$gene_name, top$gene_id)
rn <- ifelse(is.na(label_map[rows]) | !nzchar(label_map[rows]), rows, label_map[rows])
rownames(M) <- make.unique(rn)
Z <- t(scale(t(M)))

row_hc <- hclust(dist(Z),       method = "average")
col_hc <- hclust(dist(t(Z)),    method = "average")
long <- as.data.frame(as.table(Z))
names(long) <- c("gene", "sample", "z")
long$gene   <- factor(long$gene,   levels = rownames(Z)[row_hc$order])
long$sample <- factor(long$sample, levels = colnames(Z)[col_hc$order])

ggplot(long, aes(sample, gene, fill = z)) +
  geom_tile() +
  scale_fill_gradient2(low = "#2c7bb6", mid = "white", high = "#d7191c", midpoint = 0) +
  labs(x = NULL, y = NULL, fill = "z-score") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.text.y = element_text(size = 9))

In [ ]:
de_tbl |>
  dplyr::arrange(padj) |>
  dplyr::select(dplyr::any_of(c("gene_id", "gene_name", "baseMean",
                                 "log2FoldChange", "lfcSE", "stat",
                                 "pvalue", "padj"))) |>
  head(30)

## 4. GSEA

컬렉션별 Top 10 up / Top 10 down NES 바 플롯.
Ranking metric: Wald statistic

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 6)
gsea_all <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))

.coll_entries <- cfg$enrichment$gsea$collections %||% list()
.coll_labels <- setNames(
  vapply(.coll_entries, function(x) x$label %||% x$id, character(1)),
  vapply(.coll_entries, function(x) x$id, character(1))
)
.collection_label <- function(id) {
  out <- unname(.coll_labels[id])
  ifelse(is.na(out), id, out)
}

.trunc <- function(x, n = 50) ifelse(nchar(x) > n, paste0(substr(x, 1, n-1), "..."), x)

if (is.null(gsea_all) || nrow(gsea_all) == 0) {
  no_results("GSEA 결과 없음.")
} else {
  for (coll in unique(as.character(gsea_all$collection))) {
    df <- gsea_all |> dplyr::filter(collection == coll)
    if (nrow(df) == 0) next
    grp <- .collection_label(coll)
    ranked <- df |>
      dplyr::mutate(direction = dplyr::case_when(NES > 0 ~ "up", NES < 0 ~ "down", TRUE ~ NA_character_)) |>
      dplyr::filter(direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(padj, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(label = .trunc(pathway))
    ranked$label <- factor(ranked$label, levels = rev(unique(ranked$label)))
    print(
      ggplot(ranked, aes(NES, label, fill = direction)) +
        geom_col() +
        scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
        geom_vline(xintercept = 0, colour = "grey40") +
        labs(x = "NES", y = NULL,
             title = sprintf("%s — Top 10 up / Top 10 down (by padj)", grp)) +
        theme_minimal(base_size = 11) +
        theme(axis.text.y = element_text(size = 9))
    )
  }
}

### Enrichment plot

**setup** 셀을 실행한 뒤, **plot** 셀의 pathway 이름을 바꿔 실행.

In [ ]:
## --- GSEA enrichment-plot 헬퍼 (한 번만 실행) ----------------------------
suppressPackageStartupMessages({ library(fgsea); library(msigdbr) })

.ranking <- cfg$enrichment$gsea$ranking %||% "stat"
.species <- cfg$enrichment$species      %||% "Homo sapiens"

.gsea_combined <- safe_read(rp("results", "enrichment", CONTRAST, "gsea_combined.csv"))

.build_ranks <- function(de_res, method) {
  if (method == "stat") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(stat)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(stat), .groups = "drop")
  } else if (method == "signed_p") {
    rdf <- de_res |> dplyr::filter(!is.na(gene_name), !is.na(log2FoldChange), !is.na(pvalue)) |>
      dplyr::mutate(metric = sign(log2FoldChange) * -log10(pmax(pvalue, 1e-300))) |>
      dplyr::filter(is.finite(metric)) |>
      dplyr::group_by(gene_name) |> dplyr::summarise(metric = mean(metric), .groups = "drop")
  } else stop("Unsupported ranking: ", method)
  rdf <- dplyr::arrange(rdf, dplyr::desc(metric))
  setNames(rdf$metric, rdf$gene_name)
}

.get_pathways <- function(coll_str, species) {
  parts    <- unlist(strsplit(coll_str, ":"))
  coll     <- parts[1]
  subcoll  <- if (length(parts) > 1) paste(parts[-1], collapse = ":") else NULL
  m <- if (!is.null(subcoll)) msigdbr(species = species, collection = coll, subcollection = subcoll)
       else                   msigdbr(species = species, collection = coll)
  split(m$gene_symbol, m$gs_name)
}

.pathway_cache <- new.env(parent = emptyenv())

.detect_collection <- function(pathway) {
  # Fast path: MSigDB 관례적 prefix 매칭.
  prefixes <- c(HALLMARK_ = "H", REACTOME_ = "C2:CP:REACTOME",
                WP_ = "C2:CP:WIKIPATHWAYS", PID_ = "C2:CP:PID",
                BIOCARTA_ = "C2:CP:BIOCARTA")
  for (p in names(prefixes)) if (startsWith(pathway, p)) return(unname(prefixes[[p]]))
  # Slow path: 통합 GSEA 결과에서 조회.
  if (!is.null(.gsea_combined)) {
    hit <- .gsea_combined[.gsea_combined$pathway == pathway, "collection", drop = TRUE]
    if (length(hit) > 0) return(as.character(hit[1]))
  }
  stop(sprintf("'%s'의 collection을 추론할 수 없음. collection = 인자로 직접 지정하세요.", pathway))
}

.ranks <- .build_ranks(de_tbl, .ranking)

# 공개 헬퍼 -----------------------------------------------------------------

plot_gsea <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.pathway_cache[[collection]]))
    .pathway_cache[[collection]] <- .get_pathways(collection, .species)
  sets <- .pathway_cache[[collection]]
  if (!pathway %in% names(sets))
    stop(sprintf("'%s'는 %s 컬렉션에 없음.", pathway, collection))
  genes <- sets[[pathway]]
  row <- if (!is.null(.gsea_combined))
    .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  else NULL
  nes_padj <- if (!is.null(row) && nrow(row) == 1)
    sprintf(" · NES=%.2f · padj=%.2g", row$NES, row$padj) else ""
  options(repr.plot.width = 8, repr.plot.height = 5)
  fgsea::plotEnrichment(genes, .ranks) +
    ggplot2::labs(title = pathway,
                  subtitle = sprintf("%s · %s · ranking=%s · |set ∩ ranked|=%d%s",
                                     CONTRAST, collection, .ranking,
                                     sum(genes %in% names(.ranks)), nes_padj))
}

list_pathways <- function(query = NULL, n = 20) {
  if (is.null(.gsea_combined) || nrow(.gsea_combined) == 0) return(.gsea_combined)
  all <- .gsea_combined |>
    dplyr::select(collection, pathway, NES, padj, size)
  if (!is.null(query))
    all <- all[grepl(query, all$pathway, ignore.case = TRUE), ]
  dplyr::arrange(all, padj) |> head(n)
}

show_leading <- function(pathway, collection = NULL) {
  if (is.null(collection)) collection <- .detect_collection(pathway)
  if (is.null(.gsea_combined)) { message("GSEA combined CSV가 없음."); return(invisible()) }
  row <- .gsea_combined[.gsea_combined$collection == collection & .gsea_combined$pathway == pathway, ]
  if (nrow(row) == 0) { message("저장된 GSEA 결과에 해당 pathway가 없음."); return(invisible()) }
  le <- strsplit(as.character(row$leadingEdge[1]), ";", fixed = TRUE)[[1]]
  cat(sprintf("%s · %s\nNES=%.3f · padj=%.3g · size=%d · leadingEdge (%d):\n\n",
              pathway, collection, row$NES, row$padj, row$size, length(le)))
  cat(paste(le, collapse = ", "), "\n")
}

invisible(NULL)  # setup 셀은 출력 없이 조용히

In [ ]:
## --- 플롯 (pathway 이름 수정 후 실행) ----------------------------------
plot_gsea("HALLMARK_INTERFERON_GAMMA_RESPONSE")

# Collection은 이름 prefix로 자동 감지. 모호할 때는 명시:
# plot_gsea("MY_CUSTOM_SET", collection = "C2:CGP")

In [ ]:
## --- 현재 contrast에서 실제 나온 pathway 조회 / 검색 --------------------
# 모든 컬렉션 통합 Top 20 (padj 순):
list_pathways()

# 키워드 필터 (대소문자 무시):
# list_pathways("interferon")
# list_pathways("apop", n = 50)

# 특정 pathway의 leading-edge 유전자:
# show_leading("HALLMARK_INTERFERON_GAMMA_RESPONSE")

## 5. ORA

`ora_combined.csv` 기준 데이터베이스별 Top 10 up / Top 10 down (p.adjust 순).

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 7)
ora <- safe_read(rp("results", "enrichment", CONTRAST, "ora_combined.csv"))

.db_entries <- cfg$enrichment$ora$databases %||% list()
.db_labels <- setNames(
  vapply(.db_entries, function(x) x$label %||% x$id, character(1)),
  vapply(.db_entries, function(x) x$id, character(1))
)
.database_label <- function(id) {
  out <- unname(.db_labels[id])
  ifelse(is.na(out), id, out)
}

if (is.null(ora) || nrow(ora) == 0) {
  no_results("ORA 결과 없음.")
} else {
  for (db in unique(ora$database)) {
    show_df <- ora |>
      dplyr::filter(database == db, direction %in% c("up", "down")) |>
      dplyr::group_by(direction) |>
      dplyr::arrange(p.adjust, .by_group = TRUE) |>
      dplyr::slice_head(n = 10) |>
      dplyr::ungroup() |>
      dplyr::mutate(
        neglog10p   = -log10(pmax(p.adjust, 1e-300)),
        Description = ifelse(is.na(Description) | !nzchar(Description), ID, Description),
        Description = .trunc(Description, 50))
    if (nrow(show_df) == 0) next
    print(
      ggplot(show_df, aes(neglog10p, reorder(Description, neglog10p), fill = direction)) +
        geom_col(position = position_dodge(width = 0.8)) +
        scale_fill_manual(values = c(up = "#c0392b", down = "#2c7bb6")) +
        labs(x = "-log10(p.adjust)", y = NULL,
             title = sprintf("%s — Top 10 up / Top 10 down", .database_label(db))) +
        theme_minimal(base_size = 11)
    )
  }
}

## 6. TFEA (CollecTRI / ULM)

`results/tfea/<contrast>/tf_top.tsv`에서 |score| 상위 TF.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 8)
tf <- safe_read(rp("results", "tfea", CONTRAST, "tf_top.tsv"), sep = "\t")
if (is.null(tf) || nrow(tf) == 0) {
  no_results("TFEA 결과 없음.")
} else {
  top <- tf |> dplyr::slice_max(abs(score), n = 30) |>
    dplyr::mutate(tf = forcats::fct_reorder(source, score))
  ggplot(top, aes(score, tf, fill = score > 0)) +
    geom_col() +
    scale_fill_manual(values = c(`TRUE` = "#c0392b", `FALSE` = "#2c7bb6"), guide = "none") +
    labs(title = paste("Top TFEA —", CONTRAST), y = NULL) +
    theme_minimal(base_size = 11)
}

## 7. Pathway activity (PROGENy)

샘플별 signature 활성도. `results/progeny/<contrast>/progeny_scores.tsv` Heatmap은 pathway 별 z-score, 행은 contrast 그룹 간 |Δ| 순으로 정렬.

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 6)
pw <- safe_read(rp("results", "progeny", CONTRAST, "progeny_scores.tsv"), sep = "\t")

crow <- contrast_row(CONTRAST)
num  <- if (is.null(crow)) NA else as.character(crow$numerator)
den  <- if (is.null(crow)) NA else as.character(crow$denominator)

pw_mat <- NULL; sample_cond <- NULL

if (is.null(pw) || nrow(pw) == 0 || ncol(pw) < 2 ||
    all(vapply(pw[, -1, drop = FALSE], function(x) all(is.na(x)), logical(1)))) {
  no_results("PROGENy scores unavailable or all-NA.")
} else {
  pw_mat <- as.matrix(pw[, -1, drop = FALSE])
  rownames(pw_mat) <- as.character(pw[[1]])
  z_mat <- scale(pw_mat); z_mat[is.na(z_mat)] <- 0

  cond_lookup <- setNames(as.character(samples$condition), as.character(samples$sample))
  sample_cond <- cond_lookup[rownames(z_mat)]
  num_rows <- which(sample_cond == num); den_rows <- which(sample_cond == den)
  order_metric <- if (length(num_rows) > 0 && length(den_rows) > 0) {
    abs(colMeans(z_mat[num_rows, , drop = FALSE], na.rm = TRUE) -
        colMeans(z_mat[den_rows, , drop = FALSE], na.rm = TRUE))
  } else apply(z_mat, 2, function(x) diff(range(x, na.rm = TRUE)))
  path_order <- names(sort(order_metric, decreasing = TRUE))

  long <- as.data.frame(as.table(z_mat))
  names(long) <- c("sample", "pathway", "z")
  long$sample  <- factor(long$sample,  levels = rownames(z_mat))
  long$pathway <- factor(long$pathway, levels = rev(path_order))

  lim <- max(abs(long$z), na.rm = TRUE); if (!is.finite(lim) || lim == 0) lim <- 1
  ggplot(long, aes(sample, pathway, fill = z)) +
    geom_tile(colour = "white", linewidth = 0.25) +
    scale_fill_gradient2(low = "#2c7bb6", mid = "white", high = "#d7191c",
                         midpoint = 0, limits = c(-lim, lim), name = "z-score") +
    labs(x = NULL, y = NULL,
         title = "PROGENy activity, z-scored per pathway",
         subtitle = "Rows sorted by |Δ| between contrast groups") +
    theme_minimal(base_size = 12) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
}

#### Treated vs control delta + Wilcoxon p

In [ ]:
if (is.null(pw_mat) || is.na(num) || is.na(den)) {
  no_results("Delta table not computable (missing scores or contrast).")
} else {
  num_idx <- which(sample_cond == num)
  den_idx <- which(sample_cond == den)
  if (!length(num_idx) || !length(den_idx)) {
    no_results("Contrast groups not found in score table.")
  } else {
    stats_df <- do.call(rbind, lapply(colnames(pw_mat), function(path) {
      a <- pw_mat[num_idx, path]; b <- pw_mat[den_idx, path]
      delta   <- mean(a, na.rm = TRUE) - mean(b, na.rm = TRUE)
      pool_sd <- stats::sd(c(a, b), na.rm = TRUE)
      eff     <- if (is.finite(pool_sd) && pool_sd > 0) delta / pool_sd else NA_real_
      p       <- tryCatch(stats::wilcox.test(a, b, exact = FALSE)$p.value,
                          error = function(e) NA_real_)
      data.frame(pathway = path, delta = delta, effect_size = eff, wilcox_p = p)
    }))
    stats_df$wilcox_padj <- stats::p.adjust(stats_df$wilcox_p, method = "BH")
    stats_df[order(abs(stats_df$delta), decreasing = TRUE), ]
  }
}

## 8. cMap (L2S2)

`results/cmap/<contrast>/l2s2_hits.tsv`에서 |score|값 상위 perturbagen.

In [ ]:
options(repr.plot.width = 8.5, repr.plot.height = 8)
hits <- safe_read(rp("results", "cmap", CONTRAST, "l2s2_hits.tsv"), sep = "\t")
if (is.null(hits) || nrow(hits) == 0) {
  no_results("L2S2 query returned no hits.")
} else {
  top <- hits |>
    dplyr::arrange(dplyr::desc(abs(score %||% 0)), fdr) |>
    head(30) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL, title = "Top 30 perturbagens") +
    theme_minimal(base_size = 11)
}

### Re-query L2S2 with custom top-N

L2S2 paired-enrichment API를 직접 호출해 signature 크기를 바꿔 쿼리할 수 있음. **setup** 셀을 실행한 뒤, 다음 셀에서 `top_up` / `top_down`를 조정.

In [ ]:
## --- L2S2 re-query helpers (run once) ----------------------------------
suppressPackageStartupMessages({ library(httr); library(jsonlite) })

L2S2_ENDPOINT <- Sys.getenv("L2S2_API_URL", "https://l2s2.maayanlab.cloud/graphql")

.l2s2_paired_query <- '
query PairEnrichmentQuery($genesUp: [String]!, $genesDown: [String]!,
                          $first: Int = 250, $topN: Int = 10000, $pvalueLe: Float = 0.05) {
  currentBackground {
    pairedEnrich(genesUp: $genesUp, genesDown: $genesDown,
                 first: $first, topN: $topN, pvalueLe: $pvalueLe) {
      consensusCount
      consensus { drug oddsRatio pvalue adjPvalue approved countSignificant }
    }
  }
}'

query_l2s2 <- function(top_up = 250, top_down = 250, pvalue_le = 0.05) {
  if (is.null(de_tbl)) stop("de_tbl not loaded — run the DE section first.")
  df <- de_tbl |>
    dplyr::filter(!is.na(gene_name), nzchar(as.character(gene_name)),
                  !is.na(log2FoldChange), !is.na(padj)) |>
    dplyr::arrange(padj)
  up   <- head(unique(as.character(df$gene_name[df$log2FoldChange > 0])), top_up)
  down <- head(unique(as.character(df$gene_name[df$log2FoldChange < 0])), top_down)
  if (!length(up) && !length(down)) stop("empty signature: no genes with padj/log2FC.")
  cat(sprintf("Signature: %d up / %d down — querying %s\n",
              length(up), length(down), L2S2_ENDPOINT))

  payload <- list(
    query = .l2s2_paired_query,
    variables = list(genesUp = I(up), genesDown = I(down),
                     first = as.integer(max(top_up, top_down)),
                     topN = 10000L, pvalueLe = pvalue_le)
  )
  resp <- httr::POST(L2S2_ENDPOINT, httr::content_type_json(),
                     body = jsonlite::toJSON(payload, auto_unbox = TRUE))
  httr::stop_for_status(resp)
  j <- httr::content(resp, as = "parsed", simplifyVector = FALSE)
  if (!is.null(j$errors)) stop("GraphQL errors: ",
                               jsonlite::toJSON(j$errors, auto_unbox = TRUE))

  cons <- j$data$currentBackground$pairedEnrich$consensus
  if (length(cons) == 0) { cat("No consensus hits.\n"); return(data.frame()) }
  out <- do.call(rbind, lapply(cons, function(r) data.frame(
    perturbagen = r$drug       %||% NA_character_,
    score       = r$oddsRatio  %||% NA_real_,
    pvalue      = r$pvalue     %||% NA_real_,
    fdr         = r$adjPvalue  %||% NA_real_,
    approved    = r$approved   %||% NA,
    count_sig   = r$countSignificant %||% NA_integer_,
    stringsAsFactors = FALSE
  )))
  out <- out[order(out$fdr, -out$score), , drop = FALSE]
  out$rank <- seq_len(nrow(out))
  rownames(out) <- NULL
  out[, c("rank", "perturbagen", "score", "pvalue", "fdr", "approved", "count_sig")]
}

plot_l2s2 <- function(hits, n = 30, title = NULL) {
  if (is.null(hits) || nrow(hits) == 0) { no_results("No L2S2 hits."); return(invisible()) }
  top <- head(hits, n) |>
    dplyr::mutate(label = ifelse(is.na(perturbagen) | !nzchar(perturbagen),
                                 paste0("row_", dplyr::row_number()), perturbagen),
                  label = factor(label, levels = rev(label)))
  options(repr.plot.width = 8.5, repr.plot.height = max(4, n * 0.25))
  ggplot(top, aes(score, label, fill = ifelse(score > 1, "induce", "reverse"))) +
    geom_col() +
    scale_fill_manual(values = c(induce = "#e67e22", reverse = "#2c7bb6"), name = "direction") +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    labs(x = "odds ratio (score)", y = NULL,
         title = title %||% sprintf("Top %d perturbagens", nrow(top))) +
    theme_minimal(base_size = 11)
}

invisible(NULL)

In [ ]:
## --- signature 크기 조절 (top_up / top_down 수정 후 실행) --------------
hits_custom <- query_l2s2(top_up = 100, top_down = 100)
plot_l2s2(hits_custom, n = 30, title = "Top 30 perturbagens (up=100, down=100)")

# head(hits_custom, 30)
# hits_500 <- query_l2s2(top_up = 500, top_down = 500); plot_l2s2(hits_500, n = 30)